# PG-Stack · Step 7 — Analyses for the third (page-by-page) review

| Section | Review issue |
|---|---|
| S1 | Labelled patients, cancer-positive patients and malignant lesions per label budget (issue 37) |
| S2 | Composite *U* vs single detectors (incl. kNN distance) **per centre**, and inside the cross-centre blend (issue 33, 59) |
| S3 | Sensitivity of the label-free detectors to the graph size *k* ∈ {5, 10, 20} (issues 18, A4) |
| S4 | Alternative fusion rules: fixed weights, switching, max-rank, *k* = 3/6 (issue A1) |
| S5 | Permutation tests with 5,000 permutations (GPU) for the prespecified primary families (issue 25) |
| S6 | PCA variance retained; Mahalanobis dimension and ridge sensitivity (issues 15, 19) |
| S7 | Robust (median/MAD) and leave-one-out within-patient normalisation (issue 16, masking) |
| S8 | Malignant vs histopathology-confirmed benign lesions only (verification bias, B5) |
| S0 | Software versions |
| S9 | Random-effects estimator sensitivity for the cross-centre pooled effects (DL, REML, HKSJ) (issue 43) |

**Needs** (same files as Step 6): `oof_step3.csv`, `oof_step4c.csv`, `emb_pca32.npy`, `emb_dinov2_f16.npy`, `pgstack_eval.py`, `step5b_lowlabel.npz`, `step5c_c1.npz`, `step5c_c2.npz`.
**Accelerator:** GPU T4 recommended (S5 uses PyTorch for 5,000 permutations; without a GPU it falls back to 1,000). Runtime ≈ 60–100 min on a T4 (S5 is the longest part, ≈ 40 min). Each section saves its CSV as soon as it finishes; the last cell zips everything into `step7_results.zip`.

In [ ]:
import os, sys, glob, time, gc, warnings, json
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy import stats
warnings.filterwarnings("ignore")

WORK = "/kaggle/working"
COMP_DIR = "/kaggle/input/competitions/isic-2024-challenge"
SEED, SEED_C = 42, 2026
N_BOOT = 1000
N_BOOT_RE = 500          # bootstrap resamples per centre for the random-effects variances (S9)
N_PERM_GPU, N_PERM_CPU = 5000, 1000
FRAC_C1 = [0.01, 0.02, 0.03, 0.05, 0.10, 0.20]; REP_C1 = 10
FRAC_C2 = [0.01, 0.02, 0.05, 0.10]; REP_C2 = 5
FRAC_5B = [0.01, 0.02, 0.05, 0.10]; REP_5B = 10
N_BAGS, NEG_PER_POS, MIN_POS_HOSPITAL = 3, 20, 10
N_BAGS_FULL = 5          # S7 full-data models (same recipe as Step 1)
RUN_S7 = True            # retrains three full-data LightGBM models (~10-15 min)

def locate(name):
    if os.path.exists(f"{WORK}/{name}"): return f"{WORK}/{name}"
    for d in range(1, 5):
        hits = glob.glob("/kaggle/input/" + "*/" * d + name)
        if hits: return hits[0]
    return None
NEED = ["oof_step3.csv", "oof_step4c.csv", "emb_pca32.npy", "emb_dinov2_f16.npy", "pgstack_eval.py",
        "step5b_lowlabel.npz", "step5c_c1.npz", "step5c_c2.npz"]
paths = {n: locate(n) for n in NEED}
print(json.dumps(paths, indent=1))
assert all(paths.values()), f"missing: {[k for k, v in paths.items() if v is None]}"
sys.path.insert(0, os.path.dirname(paths["pgstack_eval.py"]))
from pgstack_eval import *
OUT = lambda f: f"{WORK}/s7_{f}"

## S0 — Software versions (for the Appendix)

In [ ]:
import platform, importlib
vers = {"python": platform.python_version()}
for mod in ["numpy", "pandas", "scipy", "sklearn", "lightgbm", "xgboost", "torch", "torch_geometric", "timm"]:
    try: vers[mod] = importlib.import_module(mod).__version__
    except Exception: vers[mod] = "not installed"
S0 = pd.Series(vers, name="version"); print(S0.to_string()); S0.to_csv(OUT("S0_versions.csv"))

## Features and zero-label score (identical code to Steps 5–6, so *U* is reproduced exactly)

In [ ]:
oof = pd.read_csv(paths["oof_step3.csv"])
df = pd.read_csv(f"{COMP_DIR}/train-metadata.csv", low_memory=False)
df = oof[["isic_id", "fold"]].merge(df, on="isic_id", how="left")
assert (df.isic_id.values == oof.isic_id.values).all()
emb = np.load(paths["emb_pca32.npy"]).astype(np.float32)
assert len(emb) == len(oof)

LEAK = ["lesion_id", "iddx_full", "iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5",
        "mel_mitotic_index", "mel_thick_mm", "tbp_lv_dnn_lesion_confidence"]
NUM_RAW = [c for c in df.columns if (c.startswith("tbp_lv_") or c in ["age_approx", "clin_size_long_diam_mm"])
           and c not in LEAK and pd.api.types.is_numeric_dtype(df[c])]
CAT_COLS = ["sex", "anatom_site_general", "tbp_tile_type", "tbp_lv_location", "tbp_lv_location_simple"]

def add_lesion_features(d):
    d = d.copy(); eps = 1e-6
    d["age_approx"] = pd.to_numeric(d["age_approx"], errors="coerce")
    d["lesion_size_ratio"] = d.tbp_lv_minorAxisMM / (d.clin_size_long_diam_mm + eps)
    d["lesion_shape_index"] = d.tbp_lv_areaMM2 / (d.tbp_lv_perimeterMM ** 2 + eps)
    d["hue_contrast"] = (d.tbp_lv_H - d.tbp_lv_Hext).abs()
    d["luminance_contrast"] = (d.tbp_lv_L - d.tbp_lv_Lext).abs()
    d["lesion_color_difference"] = np.sqrt(d.tbp_lv_deltaA**2 + d.tbp_lv_deltaB**2 + d.tbp_lv_deltaL**2)
    d["border_complexity"] = d.tbp_lv_norm_border + d.tbp_lv_symm_2axis
    d["color_uniformity"] = d.tbp_lv_color_std_mean / (d.tbp_lv_radial_color_std_max + eps)
    d["position_distance_3d"] = np.sqrt(d.tbp_lv_x**2 + d.tbp_lv_y**2 + d.tbp_lv_z**2)
    d["perimeter_to_area_ratio"] = d.tbp_lv_perimeterMM / (d.tbp_lv_areaMM2 + eps)
    d["area_to_perimeter_ratio"] = d.tbp_lv_areaMM2 / (d.tbp_lv_perimeterMM + eps)
    d["lesion_visibility_score"] = d.tbp_lv_deltaLBnorm + d.tbp_lv_norm_color
    d["symmetry_border_consistency"] = d.tbp_lv_symm_2axis * d.tbp_lv_norm_border
    d["color_shape_composite"] = (d.tbp_lv_color_std_mean + d.tbp_lv_area_perim_ratio + d.tbp_lv_symm_2axis) / 3
    d["log_area"] = np.log1p(d.tbp_lv_areaMM2)
    d["normalized_lesion_size"] = d.clin_size_long_diam_mm / (d.age_approx + eps)
    d["mean_hue_difference"] = (d.tbp_lv_H + d.tbp_lv_Hext) / 2
    d["std_dev_contrast"] = np.sqrt((d.tbp_lv_deltaA**2 + d.tbp_lv_deltaB**2 + d.tbp_lv_deltaL**2) / 3)
    return d

ENG = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast", "luminance_contrast", "lesion_color_difference",
       "border_complexity", "color_uniformity", "position_distance_3d", "perimeter_to_area_ratio",
       "area_to_perimeter_ratio", "lesion_visibility_score", "symmetry_border_consistency",
       "color_shape_composite", "log_area", "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast"]
df = add_lesion_features(df)
LESION_NUM = sorted(set(NUM_RAW + ENG))
for c in CAT_COLS: df[c] = df[c].fillna("NA").astype("category")
y, folds, pid = oof.target.values, oof.fold.values, oof.patient_id.values
N_FOLDS = int(folds.max()) + 1
POS_COLS = ["tbp_lv_x", "tbp_lv_y", "tbp_lv_z", "position_distance_3d"]
TAB_COLS = [c for c in LESION_NUM if c not in POS_COLS]        # appearance/shape only, for the tabular graph
print(len(df), "lesions |", len(LESION_NUM), "lesion features")

In [ ]:
df["hospital"] = df["attribution"].fillna("unknown").astype(str)
print(df.groupby("hospital").agg(lesions=("isic_id", "size"), cancers=("target", "sum"),
                                 patients=("patient_id", "nunique")).sort_values("cancers", ascending=False))

In [ ]:
g = df.groupby("patient_id"); GLOBAL = []
for c in LESION_NUM:
    df[f"{c}_pnorm"] = (df[c] - g[c].transform("mean")) / (g[c].transform("std") + 1e-6); GLOBAL.append(f"{c}_pnorm")
df["patient_n_lesions"] = g["isic_id"].transform("size"); GLOBAL.append("patient_n_lesions")

In [ ]:
X = df[LESION_NUM].astype(np.float32); X = X.fillna(X.median()).values
tab_idx = [LESION_NUM.index(c) for c in TAB_COLS]
TAB = StandardScaler().fit_transform(X[:, tab_idx]).clip(-6, 6).astype(np.float32)
TAB10 = PCA(10, random_state=SEED).fit_transform(TAB).astype(np.float32)
XYZ = df[["tbp_lv_x", "tbp_lv_y", "tbp_lv_z"]].fillna(0).values.astype(np.float32)
SP = {"tab": TAB, "app": emb, "pos": XYZ}
n = len(df)
GP = {f"gp_{s}_k{k}_{t}": np.full(n, np.nan, np.float32) for s in SP for k in [5, 10, 20] for t in ["dist", "lof"]}
GP["gp_mahal"] = np.full(n, np.nan, np.float32); GP["gp_iforest"] = np.full(n, np.nan, np.float32)

def lof(dist, idx, k):
    d, i = dist[:, :k], idx[:, :k]; reach = np.maximum(d, d[:, -1][i])
    lrd = 1.0 / (reach.mean(1) + 1e-6); return lrd[i].mean(1) / lrd

t0 = time.time()
for j, (p, rows) in enumerate(pd.Series(np.arange(n)).groupby(pid).indices.items()):
    r = np.asarray(rows); m = len(r)
    if m < 6: continue
    km = min(20, m - 1)
    for s, Z in SP.items():
        dist, idx = NearestNeighbors(n_neighbors=km + 1).fit(Z[r]).kneighbors(Z[r]); dist, idx = dist[:, 1:], idx[:, 1:]
        for k in [5, 10, 20]:
            kk = min(k, km); GP[f"gp_{s}_k{k}_dist"][r] = dist[:, :kk].mean(1); GP[f"gp_{s}_k{k}_lof"][r] = lof(dist, idx, kk)
    if m >= 15:
        Zp = TAB10[r]; dz = Zp - Zp.mean(0); C = np.cov(Zp, rowvar=False) + 1e-3 * np.eye(10)
        GP["gp_mahal"][r] = np.sqrt(np.einsum("ij,jk,ik->i", dz, np.linalg.inv(C), dz))
    if m >= 10:
        Zi = np.c_[TAB10[r], emb[r, :10]]
        GP["gp_iforest"][r] = -IsolationForest(n_estimators=100, random_state=SEED).fit(Zi).score_samples(Zi)
    if j % 200 == 0: print(f"  patient {j} | {time.time()-t0:.0f}s")
for k, v in GP.items(): df[k] = v
key = df.patient_id.astype(str) + "|" + df.anatom_site_general.astype(str)
df["gp_pos_k10_dist_siteadj"] = df["gp_pos_k10_dist"] / (df.groupby(key)["gp_pos_k10_dist"].transform("median") + 1e-6)
PRIOR = list(GP.keys()) + ["gp_pos_k10_dist_siteadj"]
print(f"graph prior: {len(PRIOR)} features in {time.time()-t0:.0f}s")

# zero-label score: average percentile rank of the 3 strongest label-free scores
U = np.nanmean(np.stack([pd.Series(df[c]).rank(pct=True).values for c in ["gp_tab_k10_lof", "gp_mahal", "gp_iforest"]]), 0)
U = np.nan_to_num(U, nan=0.5)
print("zero-label U score: AUROC", round(roc_auc_score(y, U), 4), "| pAUC80", round(pauc_80(y, U), 4))

In [ ]:
BASE_F = LESION_NUM + CAT_COLS
FSETS = {"M0_lesion": BASE_F, "M1_global": BASE_F + GLOBAL,
         "M2_prior": BASE_F + PRIOR, "M3_global_prior": BASE_F + GLOBAL + PRIOR}
MODELS = list(FSETS)
XALL = {m: df[f] for m, f in FSETS.items()}
BASE_PARAMS = dict(objective="binary", learning_rate=0.03, num_leaves=63, max_depth=7,
                   feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1, lambda_l1=0.1, lambda_l2=1.0,
                   n_estimators=300, verbosity=-1)

def fit_predict(model, tr_rows, te_rows, seed):
    pos, neg = tr_rows[y[tr_rows] == 1], tr_rows[y[tr_rows] == 0]
    Xte = XALL[model].iloc[te_rows]; pred = np.zeros(len(te_rows))
    for b in range(N_BAGS):
        rng = np.random.default_rng(seed + b)
        sub = np.r_[pos, rng.choice(neg, min(len(neg), NEG_PER_POS * len(pos)), replace=False)]
        mcs = int(np.clip(len(sub) / 50, 5, 40))
        m = lgb.LGBMClassifier(**BASE_PARAMS, min_child_samples=mcs, random_state=seed + b).fit(XALL[model].iloc[sub], y[sub])
        pred += m.predict_proba(Xte)[:, 1] / N_BAGS
    return pred

pat = df.groupby("patient_id").agg(fold=("fold", "first"), has_pos=("target", "max"),
                                   hospital=("hospital", lambda s: s.mode().iat[0]))
rows_of = pd.Series(np.arange(n)).groupby(pid).indices

def sample_patients(ids_df, share, rng):
    chosen = []
    for hp in [0, 1]:
        ids = ids_df.index[ids_df.has_pos == hp].values
        k = int(round(share * len(ids)))
        if hp == 1: k = max(1, k)                      # at least one cancer patient
        if k > 0: chosen += list(rng.choice(ids, min(k, len(ids)), replace=False))
    return chosen

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.zeros(len(p)); run = 0
    for rank, i in enumerate(o): run = max(run, min(1.0, (len(p) - rank) * p[i])); adj[i] = run
    return adj

In [ ]:
# extra detector for the ablation: Euclidean distance to the patient's centroid in shape/colour space
cent = pd.DataFrame(TAB).groupby(pid).transform("mean").values
df["gp_tab_centroid"] = np.linalg.norm(TAB - cent, axis=1)
codes = pd.factorize(pid)[0]; P = codes.max() + 1
print("patients:", P, "| U reproduced, AUROC =", round(roc_auc_score(y, U), 4))

In [ ]:
_trapz = getattr(np, "trapezoid", None) or getattr(np, "trapz")   # numpy 1.x and 2.x
def prep(yv, s, cv):
    o = np.argsort(s, kind="mergesort"); ss = s[o]
    return (yv[o], np.r_[np.diff(ss) != 0, True], cv[o])
def _roc(pr, cnt=None):
    ys, keep, co = pr; w = np.ones(len(ys)) if cnt is None else cnt[co].astype(float)
    tps = np.r_[0, np.cumsum(w * (ys == 0))[keep]]; fps = np.r_[0, np.cumsum(w * (ys == 1))[keep]]
    return fps / fps[-1], tps / tps[-1]
def pauc_p(pr, cnt=None, min_tpr=0.8):
    fpr, tpr = _roc(pr, cnt); mf = 1 - min_tpr; stop = np.searchsorted(fpr, mf, "right")
    tp = np.append(tpr[:stop], np.interp(mf, fpr[stop-1:stop+1], tpr[stop-1:stop+1])); fp = np.append(fpr[:stop], mf)
    return _trapz(tp, fp)
def auroc_p(pr, cnt=None):
    fpr, tpr = _roc(pr, cnt); return _trapz(tpr, fpr)
def ap_p(pr, cnt=None):
    ys, _, co = pr; ys = ys[::-1]; w = np.ones(len(ys)) if cnt is None else cnt[co[::-1]].astype(float)
    tp = np.cumsum(w * ys); prec = tp / np.maximum(np.cumsum(w), 1e-12)
    return float((prec * w * ys).sum() / max((w * ys).sum(), 1e-12))
_dummy = np.zeros(len(y), int)
def pauc_w(yv, s):  return pauc_p(prep(yv, s, _dummy[:len(yv)]))
def auroc_w(yv, s): return auroc_p(prep(yv, s, _dummy[:len(yv)]))
def ap_w(yv, s):    return ap_p(prep(yv, s, _dummy[:len(yv)]))

rng_b = np.random.default_rng(SEED)
BOOT_C = [np.bincount(rng_b.integers(0, P, P), minlength=P) for _ in range(N_BOOT)]
def boot_dist(metric_p, yv, scores, cv=None, n_boot=None):
    cv = codes if cv is None else cv; preps = [prep(yv, s, cv) for s in scores]
    C = BOOT_C if n_boot is None else BOOT_C[:n_boot]
    return np.array([np.mean([metric_p(pr, c) for pr in preps]) for c in C])
def ci(a): return f"[{np.percentile(a, 2.5):+.4f}, {np.percentile(a, 97.5):+.4f}]"

_s = np.random.default_rng(0).random(len(y)) + y
print("check pAUC:", round(pauc_80(y, _s), 6), round(pauc_w(y, _s), 6), "| AUROC:", round(roc_auc_score(y, _s), 6), round(auroc_w(y, _s), 6))

In [ ]:
o4 = pd.read_csv(paths["oof_step4c.csv"]); o3 = pd.read_csv(paths["oof_step3.csv"])
assert (o4.isic_id.values == df.isic_id.values).all() and (o3.isic_id.values == df.isic_id.values).all()
L5B = dict(np.load(paths["step5b_lowlabel.npz"])); C1 = dict(np.load(paths["step5c_c1.npz"])); C2 = dict(np.load(paths["step5c_c2.npz"]))
HOSP = df.hospital.values
hosp_pos = df.groupby("hospital").target.sum(); HOSPS = sorted(hosp_pos[hosp_pos >= MIN_POS_HOSPITAL].index)

def rank_in(a, groups): return pd.Series(a).groupby(groups).rank(pct=True).values
rU_fold = rank_in(U, folds)
def blend(Uv, Sv, groups, w): return w * rank_in(Uv, groups) + (1 - w) * rank_in(Sv, groups)
def c1(q, r, k=6, Uv=None):
    Uv = U if Uv is None else Uv
    m3 = C1[f"M3|{q}|{r}"]; npos = C1[f"npos|{q}|{r}"]; w = (k / (k + npos))[folds]
    return dict(M3=m3, U=Uv, adaptive=w * rank_in(Uv, folds) + (1 - w) * rank_in(m3, folds))
def c2(h, q, r, k=6, Uv=None):
    Uv = U if Uv is None else Uv
    te = np.where(HOSP == h)[0]; m3 = C2[f"M3|{h}|{q}|{r}"]; n = C2[f"npos|{h}|{q}|{r}"][0]; w = k / (k + n)
    ru = pd.Series(Uv[te]).rank(pct=True).values; rm = pd.Series(m3).rank(pct=True).values
    return te, dict(M3=m3, U=Uv[te], adaptive=w * ru + (1 - w) * rm)
def c2_eval(q, k=6, Uv=None):
    out = []
    for h in HOSPS:
        acc = {m: [] for m in ["M3", "U", "adaptive"]}
        for r in range(REP_C2):
            te, sc = c2(h, q, r, k, Uv)
            for m in acc: acc[m].append(pauc_w(y[te], sc[m]))
        out.append({m: np.mean(v) for m, v in acc.items()})
    return pd.DataFrame(out, index=HOSPS)
print("loaded")

In [ ]:
def topk(s, k):
    r = pd.Series(s).groupby(codes).rank(ascending=False, method="first").values
    f = r <= k; tp = (f & (y == 1)).sum()
    return dict(recall=tp / y.sum(), precision=tp / f.sum(), fp_per_patient=(f & (y == 0)).sum() / P, flagged=f.sum())
def workload(s, frac):
    thr = np.quantile(s, 1 - frac); f = s >= thr
    return (f & (y == 1)).sum() / y.sum(), f.sum() / P

In [ ]:
def prank(v): return pd.Series(v).rank(pct=True).fillna(0.5).values
def wp_auroc(s, yv=None, cv=None):
    yv = y if yv is None else yv; cv = codes if cv is None else cv
    d = pd.DataFrame(dict(c=cv, y=yv, s=s)); num = den = 0.0
    for _, g in d.groupby("c"):
        if 0 < g.y.sum() < len(g):
            num += roc_auc_score(g.y, g.s) * g.y.sum(); den += g.y.sum()
    return num / den
print("U within-patient AUROC:", round(wp_auroc(U), 4))

## S1 — Labelled patients and cancers per label budget

In [ ]:
rows = []
for q in sorted(set(FRAC_5B) | set(FRAC_C1)):
    for f in range(N_FOLDS):
        cand = pat[pat.fold != f]; npos_p = int(cand.has_pos.sum()); nneg_p = len(cand) - npos_p
        kp = max(1, int(round(q * npos_p))); kn = int(round(q * nneg_p))
        rec = dict(budget=q, fold=f, training_patients=len(cand), labelled_patients=kp + kn, cancer_positive_patients=kp)
        if f"npos|{q}|0" in C1:
            rec["labelled_cancers_C1"] = np.mean([C1[f"npos|{q}|{r}"][f] for r in range(REP_C1)])
        if f"npos|{q}|0" in L5B:
            rec["labelled_cancers_5b"] = np.mean([np.atleast_1d(L5B[f"npos|{q}|{r}"])[f] for r in range(REP_5B)])
        rows.append(rec)
S1 = pd.DataFrame(rows).groupby("budget").mean(numeric_only=True).drop(columns="fold").round(2)
print(S1.to_string()); S1.to_csv(OUT("S1_label_budget_counts.csv"))
rows = []
for q in FRAC_C2:
    for h in HOSPS:
        v = [C2[f"npos|{h}|{q}|{r}"][0] for r in range(REP_C2)]
        rows.append(dict(budget=q, held_out=h, labelled_cancers=np.mean(v)))
S1c2 = pd.DataFrame(rows); print(S1c2.groupby("budget").labelled_cancers.agg(["mean", "min", "max"]).round(1).to_string())
S1c2.to_csv(OUT("S1_label_budget_counts_C2.csv"), index=False)

## S2 — Composite *U* vs single detectors, per centre
Label-free, so every centre is scored without any training. Also re-runs the cross-centre blend (C2, *k* = 6) with each detector in place of *U*.

In [ ]:
DET = {"U (prespecified)": U, "kNN distance k=10": prank(df.gp_tab_k10_dist.values), "LOF k=10": prank(df.gp_tab_k10_lof.values),
       "Mahalanobis": prank(df.gp_mahal.values), "isolation forest": prank(df.gp_iforest.values),
       "centroid distance": prank(df.gp_tab_centroid.values)}
rows = []
for h in sorted(df.hospital.unique()):
    m = (HOSP == h); cv = pd.factorize(df.patient_id.values[m])[0]
    for name, s in DET.items():
        rows.append(dict(centre=h, cancers=int(y[m].sum()), detector=name, pAUC80=pauc_w(y[m], s[m]),
                         AUROC=auroc_w(y[m], s[m]), WP_AUROC=wp_auroc(s[m], y[m], cv)))
S2 = pd.DataFrame(rows)
piv = S2.pivot_table(index=["centre", "cancers"], columns="detector", values="pAUC80").round(4)
print("pAUC80 by centre:"); print(piv.to_string())
main = S2[S2.cancers >= MIN_POS_HOSPITAL]
summ = main.groupby("detector").agg(macro_pAUC=("pAUC80", "mean"), sd_pAUC=("pAUC80", "std"), min_pAUC=("pAUC80", "min"),
                                    macro_WP_AUROC=("WP_AUROC", "mean"), sd_WP_AUROC=("WP_AUROC", "std")).round(4)
rk = main.pivot_table(index="centre", columns="detector", values="pAUC80").rank(axis=1, ascending=False).mean().round(2)
summ["mean_rank_across_centres"] = rk
print("\nAcross the six centres with >= 10 cancers:"); print(summ.sort_values("macro_pAUC", ascending=False).to_string())
S2.round(4).to_csv(OUT("S2_detectors_by_centre.csv"), index=False); summ.to_csv(OUT("S2_detectors_summary.csv"))
rows = []
for name in ["U (prespecified)", "kNN distance k=10", "LOF k=10", "Mahalanobis"]:
    for q in FRAC_C2:
        per = c2_eval(q, 6, DET[name])
        rows.append(dict(prior=name, budget=q, macro_adaptive=per.adaptive.mean(), macro_M3=per.M3.mean(), macro_prior=per.U.mean(),
                         centres_better_than_M3=int((per.adaptive > per.M3).sum())))
S2b = pd.DataFrame(rows).round(4); print(S2b.to_string(index=False)); S2b.to_csv(OUT("S2_C2_blend_by_prior.csv"), index=False)

## S3 — Graph size *k* for the label-free detectors (all three values were computed; none is selected here)

In [ ]:
rows = []
for sp in ["tab", "app", "pos"]:
    for k in [5, 10, 20]:
        for t in ["dist", "lof"]:
            s = prank(df[f"gp_{sp}_k{k}_{t}"].values)
            rows.append(dict(space=sp, k=k, score=t, pAUC80=pauc_w(y, s), AUROC=auroc_w(y, s), WP_AUROC=wp_auroc(s)))
for k in [5, 10, 20]:
    Uk = np.nan_to_num(np.nanmean(np.stack([pd.Series(df[c]).rank(pct=True).values for c in [f"gp_tab_k{k}_lof", "gp_mahal", "gp_iforest"]]), 0), nan=0.5)
    rows.append(dict(space="U", k=k, score="U with LOF k", pAUC80=pauc_w(y, Uk), AUROC=auroc_w(y, Uk), WP_AUROC=wp_auroc(Uk)))
S3 = pd.DataFrame(rows).round(4); print(S3.to_string(index=False)); S3.to_csv(OUT("S3_graph_k_sensitivity.csv"), index=False)

## S4 — Alternative fusion rules
All rules use percentile ranks within the evaluation unit (fold for C1, held-out centre for C2). *Switch*: use *U* while *n* < 6, then *S*. *Max*: the larger of the two ranks.

In [ ]:
def rules(ru, rm, n):
    out = {"M3 only": rm, "U only": ru, "max-rank": np.maximum(ru, rm), "switch (n<6: U)": ru if n < 6 else rm}
    for w in [0.25, 0.5, 0.75]: out[f"fixed w={w}"] = w * ru + (1 - w) * rm
    for k in [3, 6]: w = k / (k + n); out[f"adaptive k={k}"] = w * ru + (1 - w) * rm
    return out
rows = []
for q in FRAC_C1:
    acc = {}; rec5 = {}
    for r in range(REP_C1):
        m3 = C1[f"M3|{q}|{r}"]; npos = C1[f"npos|{q}|{r}"]; s_all = {}
        for f in range(N_FOLDS):
            te = np.where(folds == f)[0]; ru = pd.Series(U[te]).rank(pct=True).values; rm = pd.Series(m3[te]).rank(pct=True).values
            for name, v in rules(ru, rm, npos[f]).items():
                s_all.setdefault(name, np.zeros(n))[te] = v
        for name, v in s_all.items():
            acc.setdefault(name, []).append(pauc_w(y, v)); rec5.setdefault(name, []).append(topk(v, 5)["recall"])
    for name in acc: rows.append(dict(setting="C1", budget=q, rule=name, pAUC80=np.mean(acc[name]), recall5=np.mean(rec5[name])))
for q in FRAC_C2:
    acc = {}
    for h in HOSPS:
        te = np.where(HOSP == h)[0]; per = {}
        for r in range(REP_C2):
            m3 = C2[f"M3|{h}|{q}|{r}"]; nn = C2[f"npos|{h}|{q}|{r}"][0]
            ru = pd.Series(U[te]).rank(pct=True).values; rm = pd.Series(m3).rank(pct=True).values
            for name, v in rules(ru, rm, nn).items(): per.setdefault(name, []).append(pauc_w(y[te], v))
        for name, v in per.items(): acc.setdefault(name, []).append(np.mean(v))
    for name in acc: rows.append(dict(setting="C2", budget=q, rule=name, pAUC80=np.mean(acc[name]), recall5=np.nan))
S4 = pd.DataFrame(rows)
best = S4[S4.rule.isin(["M3 only", "U only"])].groupby(["setting", "budget"]).pAUC80.max().rename("best_component")
S4 = S4.merge(best, on=["setting", "budget"]); S4["margin_vs_best"] = S4.pAUC80 - S4.best_component
print(S4.pivot_table(index="rule", columns=["setting", "budget"], values="pAUC80").round(4).to_string())
summ = S4.groupby(["setting", "rule"]).margin_vs_best.agg(mean_margin="mean", worst_margin="min").round(4).unstack(0)
print("\nMargin over the better of M3 and U (positive = better):"); print(summ.to_string())
S4.round(4).to_csv(OUT("S4_fusion_rules.csv"), index=False)

## S5 — Permutation tests with 5,000 permutations (PyTorch, GPU)
Same test as Step 6 R6 (swap the two models' rank-normalised scores within a random half of the patients; one-sided *p* = (1 + #{Δ_perm ≥ Δ_obs}) / (1 + N)). The torch pAUC is checked against the NumPy implementation first.

In [ ]:
import torch
DEV = "cuda" if torch.cuda.is_available() else "cpu"
N_PERM = N_PERM_GPU if DEV == "cuda" else N_PERM_CPU
print("device:", DEV, "| permutations:", N_PERM)
Yt = torch.tensor(y, dtype=torch.float64, device=DEV); CODES_t = torch.tensor(codes, device=DEV)
def pauc_t(s, yt=Yt, mf=0.2):
    o = torch.argsort(s); ss = s[o]; ys = yt[o]
    keep = torch.cat([ss[1:] != ss[:-1], torch.ones(1, dtype=torch.bool, device=s.device)])
    z = torch.zeros(1, dtype=torch.float64, device=s.device)
    tps = torch.cat([z, torch.cumsum(1 - ys, 0)[keep]]); fps = torch.cat([z, torch.cumsum(ys, 0)[keep]])
    fpr, tpr = fps / fps[-1], tps / tps[-1]
    stop = int(torch.searchsorted(fpr, torch.tensor([mf], dtype=torch.float64, device=s.device), right=True)[0])
    x0, x1, y0, y1 = fpr[stop - 1], fpr[stop], tpr[stop - 1], tpr[stop]
    yi = y0 + (y1 - y0) * (mf - x0) / (x1 - x0) if x1 > x0 else y1
    fp = torch.cat([fpr[:stop], torch.tensor([mf], dtype=torch.float64, device=s.device)]); tp = torch.cat([tpr[:stop], yi.reshape(1)])
    return float(torch.trapezoid(tp, fp))
_s = np.random.default_rng(1).random(n) + 0.6 * y
print("check torch vs numpy pAUC:", round(pauc_t(torch.tensor(_s, device=DEV)), 6), round(pauc_w(y, _s), 6))
def perm_test_t(a_list, b_list, n_perm, seed=SEED):
    A = [torch.tensor(pd.Series(a).rank(pct=True).values, device=DEV) for a in a_list]
    B = [torch.tensor(pd.Series(b).rank(pct=True).values, device=DEV) for b in b_list]
    obs = np.mean([pauc_t(a) - pauc_t(b) for a, b in zip(A, B)])
    g = torch.Generator(device=DEV); g.manual_seed(seed); ge = 0
    for _ in range(n_perm):
        sw = (torch.rand(P, generator=g, device=DEV) < 0.5)[CODES_t]
        d = np.mean([pauc_t(torch.where(sw, b, a)) - pauc_t(torch.where(sw, a, b)) for a, b in zip(A, B)])
        ge += d >= obs
    return obs, (1 + ge) / (1 + n_perm)
t0 = time.time(); fams = []
fam = []
for q in FRAC_5B:
    d, p = perm_test_t([L5B[f"M3_global_prior|{q}|{r}"] for r in range(REP_5B)], [L5B[f"M1_global|{q}|{r}"] for r in range(REP_5B)], N_PERM)
    fam.append(dict(family="label-efficiency", test=f"M3 vs M1 @{int(q*100)}%", delta=d, p_perm=p)); print(fam[-1], f"{(time.time()-t0)/60:.1f} min")
fams.append(fam); fam = []
for q in [0.01, 0.02, 0.03]:
    sc = [c1(q, r) for r in range(REP_C1)]
    d, p = perm_test_t([s["adaptive"] for s in sc], [s["M3"] for s in sc], N_PERM)
    fam.append(dict(family="adaptive vs M3", test=f"C1 @{int(q*100)}%", delta=d, p_perm=p)); print(fam[-1], f"{(time.time()-t0)/60:.1f} min")
fams.append(fam); fam = []
for q in [0.01, 0.02, 0.05, 0.10]:
    sc = [c1(q, r) for r in range(REP_C1)]
    d, p = perm_test_t([s["adaptive"] for s in sc], [s["U"] for s in sc], N_PERM)
    fam.append(dict(family="adaptive vs U", test=f"C1 @{int(q*100)}%", delta=d, p_perm=p)); print(fam[-1], f"{(time.time()-t0)/60:.1f} min")
fams.append(fam); fam = []
for a, b in [("pgb2_G4_pgboost_v2", "pgb2_G1_global"), ("pgb2_G3_pgboost_v1", "pgb2_G1_global")]:
    d, p = perm_test_t([o4[a].values], [o4[b].values], N_PERM)
    fam.append(dict(family="full data", test=f"{a} vs {b}", delta=d, p_perm=p))
fams.append(fam)
S5 = []
for fam in fams:
    adj = holm([f["p_perm"] for f in fam])
    for f, a_ in zip(fam, adj): f["p_holm"] = a_; f["n_perm"] = N_PERM; S5.append(f)
S5 = pd.DataFrame(S5).round(5); print(S5.to_string(index=False)); print(f"resolution 1/{N_PERM+1}; {(time.time()-t0)/60:.1f} min")
S5.to_csv(OUT("S5_permutation_tests_5000.csv"), index=False)

## S6 — PCA variance retained; Mahalanobis dimension and ridge sensitivity

In [ ]:
EMB = np.load(paths["emb_dinov2_f16.npy"]).astype(np.float32)
pe = PCA(64, random_state=SEED).fit(EMB); cev = np.cumsum(pe.explained_variance_ratio_)
pt = PCA(min(TAB.shape[1], 30), random_state=SEED).fit(TAB); cvt = np.cumsum(pt.explained_variance_ratio_)
S6a = pd.DataFrame([dict(space="DINOv2 (384-d)", components=c, cum_var=cev[c - 1]) for c in [8, 16, 32, 64]] +
                   [dict(space=f"shape/colour ({TAB.shape[1]} features)", components=c, cum_var=cvt[c - 1]) for c in [5, 10, 20] if c <= len(cvt)]).round(4)
print(S6a.to_string(index=False)); S6a.to_csv(OUT("S6_pca_variance.csv"), index=False)
del EMB; gc.collect()
PCS = PCA(20, random_state=SEED).fit_transform(TAB).astype(np.float32)
groups = pd.Series(np.arange(n)).groupby(pid).indices
def mahal_score(dim, ridge):
    out = np.full(n, np.nan)
    for p_, r in groups.items():
        r = np.asarray(r)
        if len(r) < 15: continue
        Zp = PCS[r, :dim]; dz = Zp - Zp.mean(0); Cm = np.cov(Zp, rowvar=False) + ridge * np.eye(dim)
        out[r] = np.sqrt(np.einsum("ij,jk,ik->i", dz, np.linalg.inv(Cm), dz))
    return out
rows = []; t0 = time.time()
for dim in [5, 10, 20]:
    for ridge in [1e-4, 1e-3, 1e-2, 1e-1]:
        mh = mahal_score(dim, ridge); s = prank(mh)
        Uv = np.nan_to_num(np.nanmean(np.stack([pd.Series(df.gp_tab_k10_lof.values).rank(pct=True).values,
                                                pd.Series(mh).rank(pct=True).values,
                                                pd.Series(df.gp_iforest.values).rank(pct=True).values]), 0), nan=0.5)
        rows.append(dict(dim=dim, ridge=ridge, mahal_pAUC80=pauc_w(y, s), mahal_AUROC=auroc_w(y, s), U_pAUC80=pauc_w(y, Uv), U_AUROC=auroc_w(y, Uv)))
S6b = pd.DataFrame(rows).round(4); print(S6b.to_string(index=False)); print(f"({time.time()-t0:.0f}s)")
S6b.to_csv(OUT("S6_mahalanobis_sensitivity.csv"), index=False)

## S7 — Robust and leave-one-out within-patient normalisation (masking check)
Full-data five-fold models with the Step 1 recipe (all positives + 1% of negatives per bag, 5 bags): standard *z* (paper), median/MAD *z*, and leave-one-lesion-out *z* (the lesion itself is excluded from its patient's mean and SD).

In [ ]:
if RUN_S7:
    t0 = time.time(); gpat = df.groupby("patient_id")
    ROB, LOO = [], []
    for c in LESION_NUM:
        x = df[c].astype(float); med = gpat[c].transform("median"); mad = (x - med).abs().groupby(df.patient_id).transform("median")
        df[f"{c}_rob"] = (x - med) / (1.4826 * mad + 1e-6); ROB.append(f"{c}_rob")
        s1 = gpat[c].transform("sum"); s2 = (x ** 2).groupby(df.patient_id).transform("sum"); cnt = gpat[c].transform("count")
        m_ = (s1 - x) / (cnt - 1); v_ = ((s2 - x ** 2) - (cnt - 1) * m_ ** 2) / (cnt - 2)
        df[f"{c}_loo"] = (x - m_) / (np.sqrt(v_.clip(lower=0)) + 1e-6); LOO.append(f"{c}_loo")
    df[ROB + LOO] = df[ROB + LOO].replace([np.inf, -np.inf], np.nan)
    LGB_FULL = dict(objective="binary", learning_rate=0.03, num_leaves=63, max_depth=7, min_child_samples=40, feature_fraction=0.6,
                    bagging_fraction=0.8, bagging_freq=1, lambda_l1=0.1, lambda_l2=1.0, n_estimators=300, verbosity=-1)
    def train_full(feats):
        oofp = np.zeros(n)
        for f in range(N_FOLDS):
            tr, te = np.where(folds != f)[0], np.where(folds == f)[0]; pos, neg = tr[y[tr] == 1], tr[y[tr] == 0]
            for b in range(N_BAGS_FULL):
                rg = np.random.default_rng(SEED + 100 * f + b); sub = np.r_[pos, rg.choice(neg, int(0.01 * len(neg)), replace=False)]
                m = lgb.LGBMClassifier(**LGB_FULL, random_state=SEED + 100 * f + b).fit(df[feats].iloc[sub], y[sub])
                oofp[te] += m.predict_proba(df[feats].iloc[te])[:, 1] / N_BAGS_FULL
        return oofp
    BASE = LESION_NUM + CAT_COLS
    PRED = {"standard z (paper)": train_full(BASE + GLOBAL), "median/MAD z": train_full(BASE + ROB + ["patient_n_lesions"]),
            "leave-one-out z": train_full(BASE + LOO + ["patient_n_lesions"])}
    ref = boot_dist(pauc_p, y, [PRED["standard z (paper)"]]); rows = []
    for name, p_ in PRED.items():
        d = boot_dist(pauc_p, y, [p_]) - ref
        rows.append(dict(normalisation=name, pAUC80=pauc_w(y, p_), AUROC=auroc_w(y, p_), WP_AUROC=wp_auroc(p_),
                         recall5=topk(p_, 5)["recall"], delta_vs_paper=pauc_w(y, p_) - pauc_w(y, PRED["standard z (paper)"]), ci95=ci(d)))
    S7 = pd.DataFrame(rows).round(4); print(S7.to_string(index=False)); print(f"({(time.time()-t0)/60:.1f} min)")
    S7.to_csv(OUT("S7_robust_normalisation.csv"), index=False)

## S8 — Malignant vs histopathology-confirmed benign lesions only
`iddx_full` is used **only to define an analysis subset** (never as a feature). Lesions labelled benign without further diagnostic detail are treated as unbiopsied. Please check the printed value counts.

In [ ]:
idd = df["iddx_full"].astype(str)
print(idd[y == 0].value_counts().head(12).to_string())
confirmed = (y == 0) & idd.str.contains("::").values
unbiopsied = (y == 0) & ~confirmed
print(f"\nconfirmed benign: {confirmed.sum()} | unbiopsied/other benign: {unbiopsied.sum()} | malignant: {int(y.sum())}")
CAND = {"U (zero-label)": U, "kNN distance k=10": DET["kNN distance k=10"], "tab_lesion": o3.oof_tab_lesion.values,
        "tab_pnorm": o3.oof_tab_pnorm.values, "G4 + prior": o4.pgb2_G4_pgboost_v2.values}
rows = []
for name, s in CAND.items():
    for sub, mask in [("malignant vs confirmed benign", (y == 1) | confirmed), ("malignant vs other benign", (y == 1) | unbiopsied),
                      ("all lesions", np.ones(n, bool))]:
        if mask.sum() == y.sum(): continue
        rows.append(dict(model=name, subset=sub, negatives=int((mask & (y == 0)).sum()), AUROC=auroc_w(y[mask], s[mask]),
                         pAUC80=pauc_w(y[mask], s[mask])))
S8 = pd.DataFrame(rows).round(4); print(S8.to_string(index=False)); S8.to_csv(OUT("S8_confirmed_benign.csv"), index=False)

## S9 — Random-effects estimator sensitivity (cross-centre, adaptive vs M3 and vs U)

In [ ]:
def re_pool(est, var):
    est, var = np.asarray(est), np.asarray(var); k = len(est); w = 1 / var
    mu_f = (w * est).sum() / w.sum(); Q = (w * (est - mu_f) ** 2).sum()
    tau_dl = max(0.0, (Q - (k - 1)) / (w.sum() - (w ** 2).sum() / w.sum()))
    tau = tau_dl
    for _ in range(200):                                     # REML by fixed-point iteration
        ws = 1 / (var + tau); mu = (ws * est).sum() / ws.sum()
        tau_new = max(0.0, (ws ** 2 * ((est - mu) ** 2 - var)).sum() / (ws ** 2).sum() + 1 / ws.sum())
        if abs(tau_new - tau) < 1e-12: break
        tau = tau_new
    tau_reml = tau
    out = {}
    for name, t2 in [("DL", tau_dl), ("REML", tau_reml)]:
        ws = 1 / (var + t2); mu = (ws * est).sum() / ws.sum(); se = np.sqrt(1 / ws.sum())
        q_hk = (ws * (est - mu) ** 2).sum() / (k - 1); se_hk = np.sqrt(max(q_hk, 1e-12) / ws.sum()); tq = stats.t.ppf(0.975, k - 1)
        out[name] = dict(mu=mu, ci_wald=(mu - 1.96 * se, mu + 1.96 * se), ci_hksj=(mu - tq * se_hk, mu + tq * se_hk), tau2=t2)
    I2 = max(0.0, (Q - (k - 1)) / Q) if Q > 0 else 0.0
    return out, I2, est.mean()
rows = []; t0 = time.time()
for q in FRAC_C2:
    for comp in ["M3", "U"]:
        est, var = [], []
        for h in HOSPS:
            te = np.where(HOSP == h)[0]; cv = pd.factorize(df.patient_id.values[te])[0]; Ph = cv.max() + 1
            sc = [c2(h, q, r)[1] for r in range(REP_C2)]
            pa = [prep(y[te], s["adaptive"], cv) for s in sc]; pb = [prep(y[te], s[comp], cv) for s in sc]
            est.append(np.mean([pauc_w(y[te], s["adaptive"]) - pauc_w(y[te], s[comp]) for s in sc]))
            rb = np.random.default_rng(SEED + int(q * 1000)); bs = []
            for _ in range(N_BOOT_RE):
                cnt = np.bincount(rb.integers(0, Ph, Ph), minlength=Ph)
                if (cnt[cv] * y[te]).sum() == 0: continue
                bs.append(np.mean([pauc_p(a, cnt) - pauc_p(b, cnt) for a, b in zip(pa, pb)]))
            var.append(np.var(bs, ddof=1))
        res, I2, macro = re_pool(est, var)
        for meth, r_ in res.items():
            rows.append(dict(budget=q, comparison=f"adaptive vs {comp}", method=meth, pooled=r_["mu"], macro_mean=macro,
                             ci_wald=f"[{r_['ci_wald'][0]:+.4f}, {r_['ci_wald'][1]:+.4f}]", ci_hksj=f"[{r_['ci_hksj'][0]:+.4f}, {r_['ci_hksj'][1]:+.4f}]",
                             tau2=r_["tau2"], I2=I2, centres_positive=int((np.array(est) > 0).sum())))
    print(f"q={q} done ({(time.time()-t0)/60:.1f} min)")
S9 = pd.DataFrame(rows).round(4); print(S9.to_string(index=False)); S9.to_csv(OUT("S9_random_effects_sensitivity.csv"), index=False)

## Bundle

In [ ]:
import zipfile
from IPython.display import FileLink
with zipfile.ZipFile(f"{WORK}/step7_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f"{WORK}/s7_*.csv")): z.write(f, os.path.basename(f))
print(sorted(os.path.basename(f) for f in glob.glob(f"{WORK}/s7_*.csv")))
FileLink("step7_results.zip")